# LLM4SBR Colab 复现实验

这个 notebook 用于在 Google Colab 上运行当前代码包。流程包括：上传代码包、安装依赖、生成缺失的 LLM session embedding、运行 `main.py` 训练推荐模型。

建议在 Colab 菜单中选择：`Runtime -> Change runtime type -> T4 GPU`。

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 上传代码包

在本地把 `LLM4SBR-main` 文件夹压缩成 `LLM4SBR-main.zip`，然后运行下面单元上传。上传完成后会解压到 `/content/LLM4SBR-main`。

In [ ]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))
assert zip_name.endswith('.zip'), '请上传 LLM4SBR-main.zip'

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content')

PROJECT_DIR = '/content/LLM4SBR-main'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())

如果你把代码放在 Google Drive，也可以不用上传 zip，改用下面方式挂载后设置 `PROJECT_DIR`。

In [ ]:
# 可选：使用 Google Drive 中的代码包时启用
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/LLM4SBR-main'
# import os
# os.chdir(PROJECT_DIR)
# print('Current directory:', os.getcwd())

## 2. 安装依赖

In [ ]:
!pip install -q -r requirements_colab.txt

## 3. 检查数据文件

这里至少需要 `Data/ml-1m/train.txt`、`Data/ml-1m/test.txt` 和 `Search_data/ml-1m/movie_name.xlsx`。

In [ ]:
from pathlib import Path

required = [
    'Data/ml-1m/train.txt',
    'Data/ml-1m/test.txt',
    'Search_data/ml-1m/movie_name.xlsx',
    'generate_llm_embeddings.py',
    'main.py',
]
for path in required:
    print(path, 'OK' if Path(path).exists() else 'MISSING')

## 4. 生成 LLM embedding

下面会生成四个文件：

- `Search_data/ml-1m/tra_session_long_emb.xlsx`
- `Search_data/ml-1m/tra_session_short_emb.xlsx`
- `Search_data/ml-1m/tes_session_long_emb.xlsx`
- `Search_data/ml-1m/tes_session_short_emb.xlsx`

默认使用 `bert-base-uncased` 作为文本编码器，并用 top-5 item 名称向量做 intent localization。第一次运行会从 HuggingFace 下载模型。完整生成会花一些时间。

如果你已经用 Qwen 生成了 `tra_session_long.xlsx` 这类 LLM 回答文件，可以在命令中追加 `--llm-output-xlsx 路径 --text-column Value`，脚本会直接编码 LLM 回答文本。

In [ ]:
import subprocess

DATASET = 'ml-1m'
ENCODER = 'bert-base-uncased'
EMB_BATCH_SIZE = '64'
LOCALIZE_TOPK = '5'

for split in ['train', 'test']:
    for view in ['long', 'short']:
        cmd = [
            'python', 'generate_llm_embeddings.py',
            '--dataset', DATASET,
            '--split', split,
            '--view', view,
            '--model-name', ENCODER,
            '--batch-size', EMB_BATCH_SIZE,
            '--localize-topk', LOCALIZE_TOPK,
            '--overwrite',
        ]
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

## 5. 确认 embedding 行数和训练数据匹配

In [ ]:
import pickle
import pandas as pd

with open('Data/ml-1m/train.txt', 'rb') as f:
    train_data = pickle.load(f)
with open('Data/ml-1m/test.txt', 'rb') as f:
    test_data = pickle.load(f)

checks = {
    'tra long': ('Search_data/ml-1m/tra_session_long_emb.xlsx', len(train_data[0])),
    'tra short': ('Search_data/ml-1m/tra_session_short_emb.xlsx', len(train_data[0])),
    'tes long': ('Search_data/ml-1m/tes_session_long_emb.xlsx', len(test_data[0])),
    'tes short': ('Search_data/ml-1m/tes_session_short_emb.xlsx', len(test_data[0])),
}

for name, (path, expected) in checks.items():
    rows = len(pd.read_excel(path))
    print(name, rows, 'expected', expected, 'OK' if rows == expected else 'MISMATCH')

## 6. 训练模型

先跑 1 个 epoch 做 smoke test；确认可以跑通后，再把 `--epoch` 改成论文设置的 30。

In [ ]:
!python main.py --dataset ml-1m --epoch 1 --batchSize 100 --hiddenSize 100 --lr 0.001 --lr_dc 0.1 --lr_dc_step 3 --l2 1e-5 --step 1 --patience 5

In [ ]:
# 完整训练时运行这一行
# !python main.py --dataset ml-1m --epoch 30 --batchSize 100 --hiddenSize 100 --lr 0.001 --lr_dc 0.1 --lr_dc_step 3 --l2 1e-5 --step 1 --patience 5